# 04 · Refine an inverted model

Takes a finished inversion (or any GoFEM model), refines it where you need smaller cells, and
optionally adds new stations. Use it to continue an inversion with new data or a finer mesh,
starting from the model you already have instead of a uniform half-space. The resistivities
carry over: when a cell is split, its 8 children keep its material id. Optionally, the refined
cells are then smoothed so the old cells' blocks don't show in them.

* **Needs:** the model's `.tria` and `.materials` (the same iteration), its receiver file and, to
  add stations, their EDIs and the model frame from notebook 02.
* **Writes** into `work/04_refine/`: `model_refined.tria` / `.vtu` / `.materials` / `.mask` and
  `receivers_refined-projected.csv`.
* **Takes:** two to three minutes for the sample (section 7, the mapping, is over half of it).
* **GUI:** setup tab 7, *Refine Model*.

Section 7 does the other way round: it puts the model's resistivities on a **fresh mesh** built for
the new data (notebook 03 or the Mesh tab), with no refinement at all.

For the worked example there's no real inversion, so the first step makes a stand-in from
notebook 03's mesh (see `sample_data/synthetic_model.py`).

## Settings

In [ ]:
# --- the model to refine -------------------------------------------------------------------
MAKE_SAMPLE_MODEL = True     # sample only: first make the stand-in "inversion" from notebook 03's mesh
MODEL_TRIA = "work/04_refine/synthetic/model_synthetic.tria"
MODEL_MATERIALS = "work/04_refine/synthetic/model_synthetic.materials"
RECEIVERS = "work/04_refine/synthetic/receivers_projected.csv"   # the model's stations

# --- new stations (optional) ------------------------------------------------------------------
NEW_STATIONS = "sample_data/edis_infill"    # EDIs of stations to add; None for no new stations
STATION_MODE = "add"                        # "add": keep the old ones too; "replace": only the new ones
FRAME_FILE = "work/02_survey/model_frame.json"   # the frame the model was built in

# --- output ---------------------------------------------------------------------------------
OUT_DIR = "work/04_refine"
STEM = "model_refined"       # file names: <STEM>.tria etc.
MAX_CELLS = 2_000_000        # stop refining before the model grows past this
WRITE_MASK = True            # write a .mask that keeps the fixed cells (air, ocean, ...) fixed

# --- smoothing the refined cells (section 4) ------------------------------------------------------
SMOOTH = True                # Gaussian-smooth the refined cells' resistivity, so the old blocks don't show
SMOOTH_MODE = "relative"     # "relative": lengths x the size of the cell each came from; "km": fixed lengths
SMOOTH_ACROSS = 0.25         # the Gaussian's standard deviation across (0.25 relative is light; 0.5 is strong)
SMOOTH_DOWN = 0.25           # ... and down

# --- or map the model onto a fresh mesh (section 7) --------------------------------------------------
MAP_ONTO = "work/03_mesh/model.tria"   # a mesh built for the new data (its .materials beside it); None to skip
MAP_FRAME_FILE = FRAME_FILE            # that mesh's frame (model_frame.json); FRAME_FILE is the model's own
MAP_SMOOTH = 0.25                      # blur the old cells' blocks by this x their size; 0 = as they are

In [ ]:
%matplotlib inline
import os

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from mtinv_gui.backends import edi_io, mesh_plots
from mtinv_gui.backends import projection as P
from mtinv_gui.backends import refine as R
from mtinv_gui.plotting.backends import georef as G

os.makedirs(OUT_DIR, exist_ok=True)

## 0. The stand-in model (sample only)

In [ ]:
if MAKE_SAMPLE_MODEL:
    import sys
    sys.path.insert(0, "sample_data")
    from synthetic_model import make_synthetic_inversion

    made = make_synthetic_inversion("work/03_mesh", os.path.dirname(MODEL_TRIA))
    print("made", ", ".join(made.values()))

## 1. Load the model

The table summarises it. *Fixed ids* are the materials the inversion didn't change (air, ocean,
mantle): GoFEM gives inverted cells new, much larger ids, so they're the ids below the first big
jump. Check that this looks right; the mask keeps them fixed.

In [ ]:
tria = R.load_tria(MODEL_TRIA)
materials = R.read_materials(MODEL_MATERIALS)
info = R.inspect(tria, materials)
fixed_ids = info.fixed_ids
top_ids = [R.AIR_ID, *info.water_ids]       # what sits on top of the earth: air, and real-ocean water
pd.Series({
    "cells": f"{info.n_cells:,} ({info.n_air:,} air)",
    "material ids": f"{info.n_ids:,}",
    "cell sizes": f"{info.smallest_m:,.0f} to {info.largest_m:,.0f} m",
    "surface elevation": f"{round(info.elevation_m[0])} to {round(info.elevation_m[1])} m",
    "fixed ids": (f"{len(fixed_ids)}: {fixed_ids[:6]}{' ...' if len(fixed_ids) > 6 else ''}"),
    "ids missing from .materials": info.missing_ids[:5] or "none",
})

## 2. Stations

The model's own stations, plus the new ones placed with the model frame from notebook 02. A new
station with the same name as an old one, in the same place, is kept once.

In [ ]:
old_names, old_xyz = R.read_receivers(RECEIVERS) if RECEIVERS else ([], np.zeros((0, 3)))
names, xy, is_new = list(old_names), old_xyz[:, :2], np.zeros(len(old_names), dtype=bool)

if NEW_STATIONS:
    georef = G.from_frame(P.load_frame(FRAME_FILE))
    new_stations = edi_io.read_edi_headers(NEW_STATIONS)
    new_names, new_xy = R.stations_in_model_frame(new_stations, georef)
    names, xy, is_new, notes = R.merge_stations(old_names, old_xyz[:, :2], new_names, new_xy, STATION_MODE)
    for note in notes:
        print(note)
print(f"{len(names)} stations: {int(is_new.sum())} new, {len(names) - int(is_new.sum())} kept")

surf = R.surface_faces(tria, top_ids)
R.project_to_surface(surf, names, xy)        # stops here if a station is outside the model
size_before = R.station_cell_sizes(surf, xy)

The model as it is: resistivity 5 km below sea level, with the old stations (white) and the new
ones (red).

In [ ]:
def show_stations(ax):
    ax.plot(xy[~is_new, 1] / 1e3, xy[~is_new, 0] / 1e3, "kv", ms=5, mfc="w")
    ax.plot(xy[is_new, 1] / 1e3, xy[is_new, 0] / 1e3, "kv", ms=6, mfc="r")


cells = mesh_plots.mesh_cells(tria, materials)
fig, (ax_rho, ax_size) = plt.subplots(1, 2, figsize=(14, 5.5))
mesh_plots.plot_plan(cells, 5000.0, ax=ax_rho, stations=np.c_[xy, np.zeros(len(xy))], vmin=0, vmax=3.5)
mesh_plots.plot_plan(cells, 5000.0, ax=ax_size, stations=np.c_[xy, np.zeros(len(xy))], color_by="size")
show_stations(ax_rho)
show_stations(ax_size)
plt.show()

## 3. Refinement rules

A rule is a region, refined `levels` times. Distances are km, depths are **below the local
surface**, positions are (north, east) in the model frame. Shapes:

* `"stations"`: an ellipsoid around each station (`stations="new"` or `"all"`), `radius_km` wide
  and `depth_bottom_km` deep;
* `"box"` / `"ellipse"`: centred on (`center_north_km`, `center_east_km`), half-widths
  `half_north_km` / `half_east_km`, from `depth_top_km` to `depth_bottom_km`;
* `"polygon"`: `polygon_km=[[north, east], ...]`; `pnorm` above 1 gives it a rounded bottom.

`min_size_km` / `min_dz_km` stop a rule from splitting cells that are already that small, so
re-running a rule doesn't keep shrinking cells. With `levels` above 1, `buffer_km` and `taper`
make nested shells like notebook 03's survey refinement: the first shell is the region grown by
`buffer_km`, and each further one is closer and shallower.

A rule can also **coarsen**: `action="coarsen"` merges split cells back into their parents (8 into 1),
`levels` times, never past the model's original grid. With `outside=True` it works on everything
whose parent stays clear of the region (grown by `buffer_km`, down to `depth_bottom_km`): the padding
around a new survey, which in an inherited model is often still fine. `max_size_km` stops it merging
into cells wider than that. Merged inverted cells get a new id with the volume-weighted mean of their
log10 conductivity; air, sea and fixed ids only merge with their own kind.

In [ ]:
RULES = [
    # the new stations: one level, but no cells smaller than 400 m
    R.RefineRule(kind="stations", stations="new", radius_km=3.0, depth_bottom_km=3.0,
                 levels=1, min_size_km=0.4),
    # the conductor in the middle of the survey: one level, plus a 10 km wider shell around it
    R.RefineRule(kind="box", center_north_km=0.0, center_east_km=-5.0,
                 half_north_km=45.0, half_east_km=15.0, depth_top_km=5.0, depth_bottom_km=25.0,
                 levels=2, buffer_km=10.0, taper=True, min_size_km=0.9),
    # R.RefineRule(kind="polygon", polygon_km=[[30, -40], [30, 0], [-20, 20], [-20, -40]],
    #              depth_bottom_km=20.0, pnorm=2.0, levels=1, min_size_km=2.0),
    # coarsen the padding: merge cells (twice) everywhere more than 20 km outside this box, or below 60 km
    # R.RefineRule(kind="box", action="coarsen", outside=True, levels=2, center_north_km=0.0,
    #              center_east_km=0.0, half_north_km=60.0, half_east_km=60.0, buffer_km=20.0,
    #              depth_bottom_km=60.0),
]

Where the rules are, on the 5 km slice (dashed: each rule's outermost shell):

In [ ]:
def outline(rule, buffer_km=0.0):
    """(east, north) km outline of a box / ellipse / polygon rule."""
    if rule.kind == "polygon":
        n, e = R.rule_polygon(rule, buffer_km).exterior.xy
        return np.asarray(e) / 1e3, np.asarray(n) / 1e3
    hn, he = rule.half_north_km + buffer_km, rule.half_east_km + buffer_km
    if rule.kind == "box":
        e = np.array([-he, he, he, -he, -he])
        n = np.array([-hn, -hn, hn, hn, -hn])
    else:
        t = np.linspace(0, 2 * np.pi, 100)
        e, n = he * np.cos(t), hn * np.sin(t)
    return e + rule.center_east_km, n + rule.center_north_km


fig, ax = plt.subplots(figsize=(7, 6))
mesh_plots.plot_plan(cells, 5000.0, ax=ax, stations=np.c_[xy, np.zeros(len(xy))], vmin=0, vmax=3.5)
show_stations(ax)
for rule in RULES:
    if rule.kind != "stations":
        ax.plot(*outline(rule), "m-", lw=1.5)
        shells = R.rule_shells(rule)
        ax.plot(*outline(rule, shells[0][0]), "m--", lw=1)
plt.show()

## 4. Refine

Each rule runs in turn. Changed your mind? Re-run from section 1: the model is reloaded from the
file. The cell first keeps a snapshot of the model as it is, for the smoothing below.

In [ ]:
before = R.snapshot_cells(tria)          # the model before refining, for the smoothing step

new_materials = {}                       # ids made by coarsening and smoothing, for the .materials
merged_keys = []                         # cells made by coarsening: smoothing leaves them alone

for k, rule in enumerate(RULES, 1):
    station_xy = None
    if rule.kind == "stations":
        station_xy = xy[is_new] if rule.stations == "new" else xy
    print(f"rule {k} ({rule.kind})")
    result = R.apply_rule(tria, rule, surf, station_xy=station_xy, exclude_ids=top_ids, max_cells=MAX_CELLS,
                          materials=materials, fixed_ids=fixed_ids)
    new_materials.update(result.get("new_ids", {}))
    merged_keys += result.get("keys", [])
    print(f"  {result['added']:+,} cells -> {result['n_cells']:,}")

### Your own refinement

As in notebook 03: flag cells and execute. Here a model-driven example: split every cell in the
top 30 km that is more conductive than 30 ohm-m and wider than 2 km, to sharpen the conductor.
Set `DO_CUSTOM = False` to skip it.

In [ ]:
DO_CUSTOM = True

if DO_CUSTOM:
    n_flagged = 0
    for cell in tria.active_cells():
        mat = materials.get(cell.material_id)
        if cell.material_id in top_ids or mat is None or mat.conductivity < 1 / 30.0:
            continue
        a, b = cell.get_vertex(0), cell.get_vertex(7)
        if cell.center().z < 30e3 and abs(b.x - a.x) > 2e3:
            cell.refine_flag = "isotropic"
            n_flagged += 1
    tria.execute_coarsening_and_refinement()
    print(f"{n_flagged:,} cells split -> {tria.n_active_cells():,} cells")

### Balance the levels

Refining leaves cells that touch only along an edge or at a corner up to two levels apart (deal.II only keeps
*face* neighbours within one level). This splits the coarser cells until no corner joins cells more than one
level apart, as p4est meshes are. It adds very few cells and doesn't move the surface. Fresh meshes with such
jumps have run in GoFEM, so this is a precaution, not a requirement.

In [ ]:
added = R.balance_levels(tria)
print(f"{added:,} cells added -> {tria.n_active_cells():,} cells")

### Smooth the refined cells

Each refined cell still has the resistivity of the cell it came from, so the old blocks show in the
finer mesh. This filters the model *as it was before refining* with a Gaussian (on log10 resistivity,
standard deviations `SMOOTH_ACROSS` and `SMOOTH_DOWN`) and gives each refined cell the filtered value at
its centre, under a new material id of its own. Other cells, air, sea and the fixed ids keep their
values, and the result can't leave the range the model already has.

With `"relative"` lengths each cell is smoothed over a fraction of the cell it came from, so the
smoothing follows the blocks at every depth (an old 20 km cell at depth gets more than a 1 km one near
the surface). Cells the inversion pushed to its bounds are averaged with their neighbours too.

In [ ]:
smoothed = None
if SMOOTH:
    smoothed = R.smooth_refined(tria, materials, before, mode=SMOOTH_MODE, horizontal=SMOOTH_ACROSS,
                                vertical=SMOOTH_DOWN, fixed_ids=fixed_ids, exclude_keys=merged_keys)
    new_materials.update(smoothed["new_ids"])
    print(f"change in log10 resistivity: median {np.median(smoothed['change']):.2f}, "
          f"99% under {np.percentile(smoothed['change'], 99):.2f}")

## 5. Stations on the refined surface, and the files

Refining doesn't move the surface, so the old stations stay where they were. Every station is
checked against the mesh (earth at the station, air or water just above it) before anything is
written.

In [ ]:
surf = R.surface_faces(tria, top_ids)
written = R.receiver_positions(surf, names, xy)      # (north, east, depth), 0.1 m below the surface
failures = R.verify_receivers(tria, names, written, top_ids=top_ids)
if failures:
    raise ValueError("stations not just below the surface: " + "; ".join(f"{n}: {w}" for n, w in failures[:5]))
print(f"{len(names)} stations checked")

paths = R.write_refined(tria, OUT_DIR, STEM, materials_path=MODEL_MATERIALS,
                        fixed_ids=fixed_ids if WRITE_MASK else None, top_ids=top_ids,
                        names=names, written=written,
                        new_materials=new_materials or None)
for kind, path in paths.items():
    print(f"{kind:10s} {path}")

## 6. Before and after

Cell size under each station, and the refined model at the same depth: resistivity (smoothed, if
`SMOOTH`) and cell size.

In [ ]:
size_after = R.station_cell_sizes(surf, xy)
display(pd.DataFrame({"station": names, "new": is_new,
                      "cell before (m)": size_before.round(), "cell after (m)": size_after.round()})
        .sort_values("new", ascending=False).head(12))

cells = mesh_plots.mesh_cells(tria, materials)      # materials now includes the smoothed cells' ids
fig, (ax_rho, ax_size) = plt.subplots(1, 2, figsize=(14, 5.5))
mesh_plots.plot_plan(cells, 5000.0, ax=ax_rho, stations=np.c_[xy, np.zeros(len(xy))], vmin=0, vmax=3.5)
mesh_plots.plot_plan(cells, 5000.0, ax=ax_size, stations=np.c_[xy, np.zeros(len(xy))], color_by="size")
show_stations(ax_rho)
show_stations(ax_size)
plt.show()

## 7. Or: map the model onto a fresh mesh

Instead of refining the old mesh, build a new one for the new data (notebook 03 or the Mesh tab) and
give its earth cells the old model's resistivity. The new mesh is used exactly as it was built: only
its material ids change, so refinement can't be the cause of any problem with it.

* Each new earth cell (id 2 in a fresh build) gets the old model's log10 resistivity **averaged over
  its volume**: a small cell gets the old cell it sits in, a large one the average of the old cells it
  covers. `MAP_SMOOTH` also blurs the old cells' blocks, as section 4's smoothing.
* The two meshes can have different projections and origins: centres go through lon/lat.
* Depth is matched **below the surface**, so different DEMs don't put new near-surface cells into the
  old model's air (`depth="elevation"` matches absolute elevation instead).
* Air, sea and other ids keep their values, and so do cells beyond the old model.

Here the sample's stand-in model goes onto notebook 03's own mesh, so the two coincide; with real data,
`MAP_ONTO` is the new survey's mesh and `MAP_FRAME_FILE` its frame.

In [ ]:
if MAP_ONTO:
    from mtinv_gui.backends import remap as RM

    target = R.load_tria(MAP_ONTO)
    target_materials_path = MAP_ONTO.replace(".tria", ".materials")
    target_materials = R.read_materials(target_materials_path)
    mapped = RM.remap(R.load_tria(MODEL_TRIA), R.read_materials(MODEL_MATERIALS),
                      G.from_frame(P.load_frame(FRAME_FILE)),            # the model's frame
                      target, target_materials, G.from_frame(P.load_frame(MAP_FRAME_FILE)),
                      fill_ids=[2], smooth=MAP_SMOOTH, depth="surface")
    receivers = os.path.join(os.path.dirname(MAP_ONTO), "receivers_projected.csv")
    map_paths = RM.write_mapped(target, target_materials_path, mapped["new_ids"], OUT_DIR, "model_mapped",
                                receivers=receivers if os.path.isfile(receivers) else "")
    cells = mesh_plots.mesh_cells(target, target_materials)
    fig, ax = plt.subplots(figsize=(7, 6))
    mesh_plots.plot_plan(cells, 5000.0, ax=ax, vmin=0, vmax=3.5)
    ax.set_title("the model on the new mesh, 5 km below sea level")
    plt.show()

## Next

* **Data for the new stations:** select them in `gateway_mt`, with the receiver file from here.
  The station names in the data must match it.
* **Run GoFEM** with the refined (or mapped: `model_mapped.*`) model as the starting model: point the `.prm` files' model and
  materials at `model_refined.tria` / `.materials`, the mask at `model_refined.mask`, and the
  stations at `receivers_refined-projected.csv`.

**For coders:** `refine.run` does sections 1-5 in one call for the GUI. To refit the surface to a
new DEM after refining (land-only models), call
`R.refit_topography(tria, dem_path, top_ids=top_ids)` before section 5; see its docstring for
what it checks. `R.smooth_refined` needs the snapshot taken *before* any refinement, your own included.